# Candidate Recovery Metric — Unplanned Readmission

Readmission is the most conventional post-surgical outcome in the literature, which makes it the
easiest to defend and the easiest to compare against published work. It is also binary, so it
discards most of the variation the wearable metrics capture, and **it is only defined for patients
observed long enough after discharge to have been readmitted** — which is the failure mode this
notebook is mostly looking for. The audit covers coverage, the censoring problem, and the event
rate, since a rate close to zero would leave a model with almost nothing to learn from.

**Governance.** Every query aggregates inside BigQuery and returns counts, quantiles or
correlations only, so no participant-level rows enter the notebook. Unsuppressed figures stay
git-ignored while `*_suppressed.png` is committable, and the small-cell threshold must be
confirmed against current All of Us policy before anything is published.

## Shared Imports and Methods

`srp_common` resolves the pinned CDR and cohort index, creates this notebook's git-ignored figure
and cache directories, and builds the cohort plus one index date per patient. **The
metric-specific CTE is defined here too**, so the whole SQL definition this notebook runs on sits
in one place: the index visit, its discharge date, the observed follow-up after it, and the
qualifying readmissions.

#### Imports, Cohort SQL and the Readmission CTEs

Transfers and episode continuations are excluded from the readmission definition here rather than
downstream, so every count below is already on the corrected definition. Every column the
notebook reads is declared to `require_columns`, so a renamed field fails at `init()` rather than
as a BigQuery 400 partway through a run.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas

_root = Path.cwd().resolve()
while not (_root / "srp_common.py").exists():
    if _root.parent == _root:
        raise RuntimeError(
            "Repo root not found: no srp_common.py at or above "
            f"{Path.cwd().resolve()}. Run this notebook from inside the repository."
        )
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import srp_common
from srp_common import CATEGORICAL, INK, INK_MUTED, SERIES, SUPPRESS_THRESHOLD

cfg = srp_common.init("readmission")
CDR = cfg.CDR
FIGURES, CACHE_DIR = cfg.FIGURES, cfg.CACHE_DIR
cached_gbq = cfg.cached_gbq

# Which of a patient's surgery-days becomes the index date is still open --
# index_surgery/surgery_day_distribution.ipynb is what settles it. "first" is used
# until then because it is the only rule no post-operative event can influence, and
# every result below is stamped with the rule so a later change is visible.
INDEX_RULE = "first"
WINDOWS = srp_common.DEFAULT_WINDOWS

METRIC_CTE = srp_common.index_visit_cte(cfg) + f""",
    followup AS (
        SELECT
            iv.person_id,
            DATE_DIFF(MAX(op.observation_period_end_date), iv.visit_end_date, DAY)
                AS followup_days
        FROM index_visit AS iv
        JOIN `{CDR}`.observation_period AS op ON op.person_id = iv.person_id
        WHERE iv.rn = 1
        GROUP BY iv.person_id
    ),
    readmission AS (
        SELECT
            iv.person_id,
            MIN(DATE_DIFF(vo.visit_start_date, iv.visit_end_date, DAY)) AS days_to_readmission
        FROM index_visit AS iv
        JOIN `{CDR}`.visit_occurrence AS vo
          ON vo.person_id = iv.person_id
         AND vo.visit_occurrence_id != iv.visit_occurrence_id
         AND vo.visit_concept_id IN (9201, 262)
         -- A stay beginning the day after discharge, or one OMOP chains to the index
         -- stay, is a transfer or a continuation of the same episode rather than a new
         -- admission. Requiring a clear day between them is the usual rule, and
         -- preceding_visit_occurrence_id catches chained records that a date gap alone
         -- would miss. IS DISTINCT FROM keeps rows where it is NULL, which is most.
         AND vo.preceding_visit_occurrence_id IS DISTINCT FROM iv.visit_occurrence_id
         AND DATE_DIFF(vo.visit_start_date, iv.visit_end_date, DAY) > 1
        WHERE iv.rn = 1
        GROUP BY iv.person_id
    )"""

CTES = cfg.cohort_ctes(INDEX_RULE, extra=METRIC_CTE)

print(f"index rule : {INDEX_RULE}")
print("windows    : " + ", ".join(f"{k} [{lo}, {hi}]" for k, (lo, hi) in WINDOWS.items()))

# Fail here, not several minutes into a BigQuery run, if the CDR does not have
# what this notebook reads. cohort_ctes checks its own tables separately.
cfg.require_columns({
    "observation_period": ["person_id", "observation_period_end_date"],
    "visit_occurrence": ["person_id", "visit_occurrence_id", "visit_concept_id", "visit_start_date", "visit_end_date", "preceding_visit_occurrence_id"],
})

## Part 1 — Coverage, Rate and Timing

Three questions that only mean anything together. **A readmission outcome needs a discharge date
to count from and enough observed follow-up after it to count within, and patients whose
observation period ends before the horizon are censored rather than event-free** — counting them
as event-free would bias the rate downward, which is the single easiest way to get this metric
wrong. Both horizons are reported because the 30-day rate is the conventional one and the 90-day
rate is closer to "recovery" than to "early failure", and the timing distribution says which of
them is cutting through the middle of the events.

#### Who Can Be Scored, How Often It Happens, and When

The rate is computed over patients actually observed to the horizon, never over the whole cohort,
which is what keeps censoring out of the numerator and the denominator alike. **A distribution
still rising at day 30 argues for the longer horizon**; one that has flattened says the
conventional window is capturing the bulk of events.

In [ ]:
coverage_sql = CTES + """
    SELECT
        (SELECT COUNT(*) FROM cohort)                       AS cohort_patients,
        (SELECT COUNT(*) FROM index_visit WHERE rn = 1)     AS patients_with_index_visit,
        (SELECT COUNT(*) FROM followup)                     AS patients_with_followup_record,
        (SELECT COUNTIF(followup_days >= 30) FROM followup) AS followed_30d,
        (SELECT COUNTIF(followup_days >= 90) FROM followup) AS followed_90d
"""

cov = cached_gbq(coverage_sql, "readmission_coverage").iloc[0]
cohort_patients = int(cov["cohort_patients"])
followed_30d = int(cov["followed_30d"])

for label in ["patients_with_index_visit", "patients_with_followup_record",
              "followed_30d", "followed_90d"]:
    n = int(cov[label])
    print(f"{label:30s} {n:>9,}   ({n / cohort_patients:6.1%} of cohort)")

print()
print("--> followed_30d is the denominator for the 30-day rate. Anyone short of it is "
      "censored, and must not be counted as event-free.")

# The rate at both horizons, each over the patients observed that long -- NOT over the
# cohort, since a censored patient is not an event-free one.
rate_sql = CTES + """
    SELECT
        30                                                          AS horizon_days,
        COUNTIF(f.followup_days >= 30)                              AS n_observed,
        COUNTIF(f.followup_days >= 30 AND r.days_to_readmission BETWEEN 0 AND 30)
                                                                    AS n_readmitted
    FROM followup AS f
    LEFT JOIN readmission AS r ON r.person_id = f.person_id
    UNION ALL
    SELECT
        90                                                          AS horizon_days,
        COUNTIF(f.followup_days >= 90)                              AS n_observed,
        COUNTIF(f.followup_days >= 90 AND r.days_to_readmission BETWEEN 0 AND 90)
                                                                    AS n_readmitted
    FROM followup AS f
    LEFT JOIN readmission AS r ON r.person_id = f.person_id
"""

rates = cached_gbq(rate_sql, "readmission_rates").sort_values("horizon_days")
rates["rate"] = rates["n_readmitted"] / rates["n_observed"]

for _, row in rates.iterrows():
    n_events = int(row["n_readmitted"])
    flag = "  << below threshold, do not report" if 0 < n_events < SUPPRESS_THRESHOLD else ""
    print(f"{int(row['horizon_days']):>3}-day  observed={int(row['n_observed']):>8,}  "
          f"readmitted={n_events:>7,}  rate={row['rate']:6.2%}{flag}")

rate_30 = float(rates.loc[rates["horizon_days"] == 30, "rate"].iloc[0])

# Where the events actually fall, in weeks since discharge, pooled at 13+.
timing_sql = CTES + """
    SELECT
        LEAST(CAST(FLOOR(r.days_to_readmission / 7) AS INT64), 13) AS week_bucket,
        COUNT(*)                                                   AS n_patients
    FROM readmission AS r
    JOIN followup AS f ON f.person_id = r.person_id
    WHERE r.days_to_readmission BETWEEN 0 AND 90
      AND f.followup_days >= 90
    GROUP BY week_bucket
    ORDER BY week_bucket
"""

timing = cached_gbq(timing_sql, "readmission_timing")
timing["week_label"] = [
    "13+" if int(v) >= 13 else f"wk {int(v)}" for v in timing["week_bucket"]
]

srp_common.plot_distribution(
    cfg, timing, "week_label",
    title="Readmissions by week after discharge, among patients observed 90 days",
    xlabel="weeks after discharge from the index stay", suppress=True,
    save_as="readmission_timing")

## Part 2 — Verdict

The audit criteria are recorded as one row in the shared task cache, which is what
`metric_comparison.ipynb` reads to build the cross-metric scorecard. **Nothing written here is a
participant-level value**: the row is counts, fractions and hand-entered judgements.

#### The Scorecard Row

Writing the verdict as data rather than prose is what lets the cross-metric comparison be
computed instead of transcribed. The hand-entered fields are judgements and are recorded as such,
so a later reader can disagree with one specifically rather than with the whole row.

In [ ]:
srp_common.write_scorecard(
    cfg,
    family="ehr",
    source_table="visit_occurrence + observation_period",
    index_rule=INDEX_RULE,
    outcome_type="binary",
    cohort_patients=cohort_patients,
    patients_with_metric=int(cov["patients_with_index_visit"]),
    patients_analysable=followed_30d,
    coverage_share=followed_30d / cohort_patients,
    horizon_days=30,
    event_rate=rate_30,
    informative_missingness="no",
    leakage_risk="medium",
    notes=("Censoring is the handling risk: patients whose observation period ends before the "
           "horizon are not event-free. A readmission is also not necessarily related to the "
           "surgery, so this measures early failure more than recovery."),
)